# Conductance-Based LIF Pipeline (Modular)

This notebook is the recommended orchestration path for the conductance-based clustered LIF workflow. It imports the shared `lif_simulation` package instead of embedding the full simulation, loading, analysis, and plotting pipeline inline.

The original `LIF_network_simulation_network_burst_conductance.ipynb` notebook is retained as the monolithic reference/extraction notebook, while plotting and reusable workflow logic live in the package modules and thin scripts.

**Suggested first-time run order:**
1. Run the import cell.
2. Review the configuration cell and keep `execute_main_simulation = False` on the first pass.
3. Run the no-stimulation validation cell and the sag/rebound probe cell.
4. Check the printed validation verdict plus the raster and voltage plots.
5. Set `execute_main_simulation = True` only when the quick validation outputs look reasonable.
6. After a session is saved, use `session_source` and `recording_index` in the later cells to load and inspect that recording.

**How to choose the run mode:**
- For spontaneous activity, keep `burst_interval` extremely large and set `cluster_fraction = 0.0` with `neurons_per_cluster = 0`.
- For stimulus-driven bursting, use a finite `burst_interval` together with nonzero cluster stimulation settings.
- Toggle `use_h_current` in the configuration cell when you want the h-current enabled or disabled for ablation.

**What this notebook writes:**
- Saved sessions under `LIF data/<timestamp>`
- Raw full-dt voltage traces by default, stored in chunked HDF5 sidecars during saved sessions
- Shared plotting outputs built from `lif_simulation/plotting.py`

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

from lif_simulation.analysis import analyze_spike_trains, compute_hub_firing_rate_groups, validate_hub_structure
from lif_simulation.plotting import (
    plot_combined_network_layout,
    plot_firing_rates,
    plot_hub_degree_distributions,
    plot_hub_firing_rate_histogram,
    plot_hub_network,
    plot_network_positions,
    plot_raster,
    plot_resampled_raster,
    plot_sag_probe,
    plot_spike_train_analysis_summary,
    plot_voltage_heatmap,
    plot_voltage_traces,
)
from lif_simulation.probes import run_h_current_step_probe, run_no_stimulation_validation, summarize_h_current_step_probe
from lif_simulation.session_views import load_session_bundle
from lif_simulation.workflows import sequential_simulation_individual_saves

In [ ]:
seed = 41
np.random.seed(seed)
random.seed(seed)

validation_config = {
    'seed': seed,
    'use_h_current': True,
    'test_duration_ms': 12000,
    'requested_voltage_sample_rate': 0.25,
}

sag_probe_config = {
    'step_current_nA': -0.12,
    'duration_ms': 1200.0,
    'dt': 0.1,
}

execute_main_simulation = True
simulation_params = {
    'n_recordings': 20,
    'recording_duration': 60000,
    'num_clusters': 30,
    'neurons_per_cluster_range': (12, 18),
    'inhibitory_probability': 0.2,
    'within_cluster_prob': 0.3,
    'between_cluster_prob': 0.15,
    'target_freq': 10,
    'save_dir': 'LIF data',
    'dt': 0.1,
    'record_voltage': True,
    'voltage_sample_rate': 1.0,
    'voltage_storage_backend': 'hdf5_external',
    'voltage_chunk_samples': 4096,
    'space_size': 15,
    'max_connection_distance': 6.0,
    'use_h_current': True,
    'burst_interval': 12000,
    'cluster_fraction': 0.5,
    'neurons_per_cluster': 2,
    'stim_amplitude_range': (2.0, 3.5),
    'stim_duration_range': (10, 30),
    'burst_interval_jitter': 1500,
    'hub_fraction': 0.1,
    'hub_between_prob': 0.25,
    'hub_weight_scale': 1.5,
    'hub_reciprocal_factor': 2.0,
}

session_source = 'latest'
recording_index = 0

In [ ]:
validation_result = run_no_stimulation_validation(**validation_config)
validation_metrics = validation_result['metrics']

print('NO_STIM_TEST')
for key in [
    'use_h_current',
    'external_stimulation_events',
    'test_duration_s',
    'requested_voltage_sample_rate_ms',
    'saved_voltage_step_ms',
    'voltage_trace_mode',
    'raster_window_s',
    'n_neurons',
    'total_spikes',
    'active_neurons',
    'mean_rate_hz',
    'median_rate_hz',
    'max_rate_hz',
    'max_active_fraction_100ms',
    'bins_ge_10pct',
    'bins_ge_25pct',
    'bins_ge_50pct',
    'verdict',
]:
    print(f'{key}={validation_metrics[key]}')

validation_raster_spike_data = {
    neuron_id: [spike for spike in spikes if spike <= validation_result['raster_window_ms']]
    for neuron_id, spikes in validation_result['spike_data'].items()
}

plot_combined_network_layout(
    validation_result['neuron_positions'],
    validation_result['cluster_info'],
    validation_result['connections'],
    cluster_assignments=validation_result['cluster_assignments'],
    max_connections=validation_result['max_connections_plot'],
    title='No-Stimulation Test: Network Structure',
)
plt.show()

plot_raster(
    validation_raster_spike_data,
    validation_result['cluster_assignments'],
    validation_result['raster_window_ms'],
    title=f"No-Stimulation Test Raster ({validation_result['raster_window_ms'] / 1000.0:.1f} s shown)",
)
plt.show()

plot_voltage_traces(
    validation_result['voltage_data'],
    neuron_ids=validation_result['example_neuron_ids'],
    time_range=(0, validation_result['voltage_window_ms']),
    spike_data=validation_result['spike_data'],
    title=f"No-Stimulation Test Raw Voltage Traces ({validation_result['voltage_window_ms'] / 1000.0:.1f} s window)",
)
plt.show()

In [ ]:
sag_with_h = run_h_current_step_probe(use_h_current=True, **sag_probe_config)
sag_without_h = run_h_current_step_probe(use_h_current=False, **sag_probe_config)
sag_summary = summarize_h_current_step_probe(sag_with_h, sag_without_h)

print('H_CURRENT_SAG_TEST')
for key, value in sag_summary.items():
    print(f'{key}={value:.3f}' if isinstance(value, float) else f'{key}={value}')

plot_sag_probe(sag_with_h, sag_without_h)
plt.show()

In [ ]:
if execute_main_simulation:
    np.random.seed(seed)
    random.seed(seed)
    session_metadata = sequential_simulation_individual_saves(**simulation_params)
    print(f"Saved session to: {session_metadata['session_dir']}")
else:
    print('Main simulation skipped. Set execute_main_simulation = True to run it.')

In [ ]:
try:
    bundle = load_session_bundle(session_source=session_source, recording_index=recording_index)
    print(f"Loading session: {bundle['timestamp']}")
    print(f"Session folder: {bundle['session_dir']}")
    print(f"Relative session path: {bundle['relative_session_path']}")
except FileNotFoundError as exc:
    bundle = None
    print(exc)

if bundle is not None:
    activity_label = 'stimulus-driven' if bundle['stimulation_enabled'] else 'spontaneous firing'
    plot_raster(
        bundle['spike_data_dict'],
        bundle['cluster_assignments'],
        bundle['recording_duration'],
        title=f"Recording - Raster Plot ({bundle['recording_duration'] / 1000:.0f}s, {activity_label})",
    )
    plt.show()

    if bundle['voltage_data'] is not None:
        plot_voltage_traces(
            bundle['voltage_data'],
            neuron_ids=[0, 5, 10, 15, 20, 21, 22, 23, 24, 25],
            time_range=(0, 2000),
            spike_data=bundle['spike_data_dict'],
            title='Stored Voltage Traces (0-2000 ms)',
        )
        plt.show()

        plot_voltage_heatmap(
            bundle['voltage_data'],
            time_range=(0, 2000),
            cluster_assignments=bundle['cluster_assignments'],
            title='Network Voltage Heatmap (0-2 seconds)',
        )
        plt.show()

    plot_firing_rates(bundle['spike_data_dict'], bundle['cluster_assignments'], bundle['recording_duration'])
    plt.show()

    plot_network_positions(
        bundle['network_positions'],
        bundle['network_cluster_info'],
        bundle['network_connections'],
        cluster_assignments=bundle['cluster_assignments'],
        show_connections='sample',
        max_connections=500,
    )
    plt.show()

    plot_resampled_raster(
        bundle['resampled_spikes'],
        bundle['resampled_times'],
        bundle['cluster_assignments'],
        bundle['resampling_frequency'],
        burst_onset_times=bundle['burst_onset_times'],
    )
    plt.show()

    analysis_results = analyze_spike_trains(
        bundle['spike_data_dict'],
        bundle['cluster_assignments'],
        bundle['recording_duration'],
        bundle['network_connections'],
        burst_onset_times=bundle['burst_onset_times'],
    )
    plot_spike_train_analysis_summary(analysis_results)
    plt.show()

    if bundle['hub_cluster_info'] is not None:
        plot_hub_network(bundle['network_positions'], bundle['network_connections'], bundle['hub_cluster_info'])
        plt.show()

        hub_stats = validate_hub_structure(
            bundle['network_connections'],
            bundle['hub_cluster_info'],
            len(bundle['network_positions']),
        )
        if hub_stats is not None:
            plot_hub_degree_distributions(hub_stats)
            plt.show()
            rate_groups = compute_hub_firing_rate_groups(
                bundle['spike_data_dict'],
                bundle['hub_cluster_info']['hub_neuron_ids'],
                bundle['recording_duration'],
            )
            plot_hub_firing_rate_histogram(rate_groups)
            plt.show()